In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
import torch.optim as optim

# 1. Cấu hình đường dẫn
DRIVE_FOLDER_PATH = '/content/drive/MyDrive/eyepacs_6k'
EXTRACT_DIR       = '/content/dataset_full'
CSV_PATH          = '/content/full_labels.csv'
SAVE_PATH         = '/content/drive/MyDrive/efficientnet_b4_dr_best.pth'
LAST_PATH         = '/content/drive/MyDrive/efficientnet_b4_dr_last.pth'

# 2. Siêu tham số (Giữ nguyên Batch Size 256 tối ưu)
IMG_SIZE, BATCH_SIZE, NUM_EPOCHS = 380, 128, 20
LR, WEIGHT_DECAY = 3e-4, 1e-4
MIXUP_ALPHA      = 0.1
PATIENCE, VAL_RATIO, NUM_CLASSES, SEED = 7, 0.15, 5, 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Đang sử dụng thiết bị: {device}")

torch.manual_seed(SEED)
np.random.seed(SEED)

def extract_from_drive_folder(drive_folder, extract_dir):
    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 5:
        print(f" Dữ liệu ảnh đã có sẵn trên SSD, bỏ qua giải nén.")
        return

    print(f" Bước 1: Quét thư mục '{drive_folder}' trên Drive...")
    os.makedirs(extract_dir, exist_ok=True)

    csv_files = glob.glob(os.path.join(drive_folder, '*.csv'))
    if csv_files:
        shutil.copy(csv_files[0], CSV_PATH)
        print(f"    Đã trích xuất file CSV: {os.path.basename(csv_files[0])}")
    else:
        print("   LỖI: Không tìm thấy file .csv!")
        return

    sub_zips = glob.glob(os.path.join(drive_folder, '*.zip'))
    print(f"  Bắt đầu gộp {len(sub_zips)} file zip con...")

    local_temp_zip = '/content/temp_data.zip'
    for i, zip_path in enumerate(sub_zips, 1):
        print(f"      [{i}/{len(sub_zips)}] Đang bung: {os.path.basename(zip_path)}", end='\r')
        shutil.copyfile(zip_path, local_temp_zip)
        !unzip -q -n {local_temp_zip} -d {extract_dir}
        os.remove(local_temp_zip)

    print("\n HOÀN TẤT GIẢI NÉN VÀ GỘP DỮ LIỆU!")

extract_from_drive_folder(DRIVE_FOLDER_PATH, EXTRACT_DIR)

 Đang sử dụng thiết bị: cuda
 Bước 1: Quét thư mục '/content/drive/MyDrive/eyepacs_6k' trên Drive...
    Đã trích xuất file CSV: trainLabels.csv
  Bắt đầu gộp 6 file zip con...


In [ ]:
# BLOCK 2: CHUẨN BỊ DATALOADER (HỌC TỰ NHIÊN)
def ben_color(path, sigma=10, size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    mask = np.zeros(img.shape, dtype=np.uint8)
    cv2.circle(mask, (size // 2, size // 2), int(size * 0.47), (1, 1, 1), -1)
    return Image.fromarray((img * mask).astype(np.uint8))

class EyePacsDataset(Dataset):
    def __init__(self, records, image_paths, transform=None):
        self.records, self.image_paths, self.transform = records, image_paths, transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        img_name, label = self.records[idx]
        img_id = os.path.splitext(img_name)[0]
        if img_id not in self.image_paths: return None
        image = ben_color(self.image_paths[img_id])
        if image is None: return None
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    return torch.utils.data.dataloader.default_collate(batch) if batch else None

print(" Đang lập bản đồ đường dẫn ảnh...")
image_paths = {os.path.splitext(f)[0]: os.path.join(root, f)
               for root, _, files in os.walk(EXTRACT_DIR)
               for f in files if f.lower().endswith(('.jpeg', '.jpg', '.png'))}

train_tfm = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tfm = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

df = pd.read_csv(CSV_PATH)
records = [(str(row.iloc[0]).strip(), int(row.iloc[1])) for _, row in df.iterrows()]
labels = [r[1] for r in records]

train_rec, val_rec = train_test_split(records, test_size=VAL_RATIO, stratify=labels, random_state=SEED)
print(f" Train: {len(train_rec):,} ảnh  |  Validation: {len(val_rec):,} ảnh")

train_loader = DataLoader(EyePacsDataset(train_rec, image_paths, train_tfm), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=safe_collate, pin_memory=True)
val_loader   = DataLoader(EyePacsDataset(val_rec, image_paths, val_tfm), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=safe_collate, pin_memory=True)
print(" Hoàn tất thiết lập DataLoader!")

In [ ]:
# BLOCK 3: KHỞI TẠO MODEL EFFICIENTNET-B4
model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if any(k in name for k in ['features.6', 'features.7', 'features.8', 'classifier']):
        param.requires_grad = True

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Kiến trúc: EfficientNet-B4 | Tham số mở khóa để train: {total_params:,}")

In [ ]:
# BLOCK 4: CÔNG CỤ TỐI ƯU & NẠP MODEL CŨ ĐỂ RESUME
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda')

# Khai báo biến toàn cục để Block 5 đọc tiến trình
START_EPOCH = 0
best_qwk    = -1.0

# Bạn chọn đường dẫn muốn nạp
RESUME_PATH = SAVE_PATH

if os.path.exists(RESUME_PATH):
    print(f" Tìm thấy mô hình cũ tại: {RESUME_PATH}")

    # ĐÃ FIX LỖI Ở ĐÂY: Thêm weights_only=False để cho phép nạp biến numpy của best_qwk
    checkpoint = torch.load(RESUME_PATH, map_location=device, weights_only=False)

    # Trường hợp 1: File lưu đầy đủ cấu trúc từ điển
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        model.load_state_dict(checkpoint['model_state'])

        if 'optim_state' in checkpoint:
            optimizer.load_state_dict(checkpoint['optim_state'])
            print(" Đã phục hồi trạng thái bộ tối ưu AdamW (giữ vững quán tính học).")

        START_EPOCH = checkpoint['epoch']
        best_qwk    = checkpoint['best_qwk']

        # Đồng bộ hóa lại lịch trình giảm Learning Rate của Scheduler
        for _ in range(START_EPOCH):
            scheduler.step()

        print(f" NẠP THÀNH CÔNG TOÀN DIỆN!")
        print(f" Sẽ tiếp tục huấn luyện tiếp nối từ Epoch: {START_EPOCH + 1}")
        print(f" Điểm QWK đỉnh cao trước đó: {best_qwk:.4f}")

    # Trường hợp 2: File chỉ lưu state_dict thuần túy
    else:
        model.load_state_dict(checkpoint)
        print(f" Đã nạp Trọng số mô hình thuần túy. Bộ tối ưu và lịch trình sẽ học lại từ đầu.")
else:
    print(" Không tìm thấy mô hình cũ trên Drive. Hệ thống sẽ huấn luyện mới tinh từ ImageNet.")

# --- ĐỊNH NGHĨA CÁC HÀM TRAIN ---
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    for i, batch in enumerate(loader):
        if batch is None: continue
        images, labels = batch[0].to(device), batch[1].to(device)
        images, targets_a, targets_b, lam = mixup_data(images, labels)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if (i + 1) % 50 == 0:
            print(f"  - Batch [{i+1}/{len(loader)}] | Loss = {loss.item():.4f}")

    scheduler.step()
    return running_loss / len(all_labels), accuracy_score(all_labels, all_preds), cohen_kappa_score(all_labels, all_preds, weights='quadratic')

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, all_preds, all_labels = 0.0, [], []
    for batch in loader:
        if batch is None: continue
        images, labels = batch[0].to(device), batch[1].to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    print("\n   [Ma trận nhầm lẫn - Validation]:")
    print(confusion_matrix(all_labels, all_preds))
    return running_loss / len(all_labels), accuracy_score(all_labels, all_preds), cohen_kappa_score(all_labels, all_preds, weights='quadratic')

print("\n Thiết lập hàm tối ưu + AMP và cấu trúc Resume hoàn tất!")

In [ ]:
# BLOCK 5: VÒNG LẶP HUẤN LUYỆN CHÍNH (RESUME LOOP)
patience_cnt = 0
history = []

print(f"\n KÍCH HOẠT CHIẾN DỊCH HUẤN LUYỆN TIẾP NỐI (TỪ EPOCH {START_EPOCH + 1})...\n")

for epoch in range(START_EPOCH, NUM_EPOCHS):
    print(f"═══ Epoch [{epoch+1}/{NUM_EPOCHS}] ═══")

    tr_loss, tr_acc, tr_qwk = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
    vl_loss, vl_acc, vl_qwk = evaluate(model, val_loader, criterion)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n  TRAIN → Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}%  QWK: {tr_qwk:.4f}")
    print(f"  VAL   → Loss: {vl_loss:.4f}  Acc: {vl_acc*100:.2f}%  QWK: {vl_qwk:.4f}  lr: {current_lr:.2e}")

    history.append(dict(epoch=epoch+1, tr_loss=tr_loss, tr_acc=tr_acc, tr_qwk=tr_qwk, vl_loss=vl_loss, vl_acc=vl_acc, vl_qwk=vl_qwk))

    if vl_qwk > best_qwk:
        best_qwk, patience_cnt = vl_qwk, 0
        torch.save({
            'epoch':       epoch + 1,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'best_qwk':    best_qwk
        }, SAVE_PATH)
        print(f"   Đã phá đỉnh cũ! Đã lưu Best Model mới vào Drive! QWK={best_qwk:.4f}")
    else:
        patience_cnt += 1
        print(f"   Không cải thiện QWK ({patience_cnt}/{PATIENCE})")

    torch.save(model.state_dict(), LAST_PATH)
    print(f" Đã lưu backup an toàn cuối Epoch vào Drive!\n")

    if patience_cnt >= PATIENCE:
        print(f" Kích hoạt Early Stopping. Dừng tại Epoch {epoch+1}. Best QWK={best_qwk:.4f}")
        break

print("\n KẾT QUẢ ĐỈNH CAO SAU KHI TIẾP TỤC TRAIN: QWK =", f"{best_qwk:.4f}")